In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
import requests, zipfile, io, os

FIRE_CSV   = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/climate-pressure/fire/DL_FIRE_J1VIIRS-C2_718929_Nov2000-Feb2026_buffer0km-csv/fire_archive_J1V-C2_718929.csv"   # VIIRS fire data
ACLED_CSV  = "/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx"               # ACLED conflict data

# ---------- 1. Download Indonesia district shapefile (GADM Level 1) ----------
print("Downloading Indonesia shapefile...")
gadm_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IDN_shp.zip"
r = requests.get(gadm_url, timeout=60)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("gadm_idn")
idn = gpd.read_file("gadm_idn/gadm41_IDN_1.shp")  # Province level
print(f"Shapefile loaded: {len(idn)} provinces")

# ---------- 2. Load and clean fire data ----------
print("Loading fire data...")
fire = pd.read_csv(FIRE_CSV)
fire = fire.rename(columns={"latitude": "lat", "longitude": "lon"})
fire = fire.dropna(subset=["lat", "lon"])
# Filter to Indonesia bounding box
fire = fire[(fire["lat"].between(-11, 6)) & (fire["lon"].between(95, 141))]
print(f"Fire records after filtering: {len(fire):,}")

# ---------- 3. Load and clean ACLED data ----------
print("Loading ACLED data...")
acled = pd.read_csv(ACLED_CSV)

# Detect lat/lon columns (handles both naming conventions)
lat_col = next((c for c in acled.columns if "centroid_lat" in c.lower() or c.lower() == "latitude"), None)
lon_col = next((c for c in acled.columns if "centroid_lon" in c.lower() or c.lower() == "longitude"), None)
event_col = next((c for c in acled.columns if "event_type" in c.lower()), "EVENT_TYPE")
fatal_col = next((c for c in acled.columns if "fatal" in c.lower()), None)

acled = acled.rename(columns={lat_col: "lat", lon_col: "lon", event_col: "event_type"})
acled = acled.dropna(subset=["lat", "lon"])
acled = acled[(acled["lat"].between(-11, 6)) & (acled["lon"].between(95, 141))]
print(f"ACLED records after filtering: {len(acled):,}")

# ---------- 4. Build fire KDE heatmap grid ----------
print("Building fire KDE heatmap...")
x_fire = fire["lon"].values
y_fire = fire["lat"].values
# Subsample for KDE performance if very large
if len(x_fire) > 200_000:
    idx = np.random.choice(len(x_fire), 200_000, replace=False)
    x_fire, y_fire = x_fire[idx], y_fire[idx]

xgrid = np.linspace(95, 141, 500)
ygrid = np.linspace(-11, 6, 300)
XX, YY = np.meshgrid(xgrid, ygrid)
positions = np.vstack([XX.ravel(), YY.ravel()])
values    = np.vstack([x_fire, y_fire])
kernel    = gaussian_kde(values, bw_method=0.05)
ZZ = kernel(positions).reshape(XX.shape)

# ---------- 5. ACLED event type colors ----------
event_colors = {
    "Battles":                    "#d62728",
    "Violence against civilians": "#ff7f0e",
    "Protests":                   "#2ca02c",
    "Riots":                      "#9467bd",
    "Explosions/Remote violence": "#8c564b",
    "Strategic developments":     "#e377c2",
}
default_color = "#7f7f7f"

def get_color(etype):
    for key in event_colors:
        if isinstance(etype, str) and key.lower() in etype.lower():
            return event_colors[key]
    return default_color

acled["color"] = acled["event_type"].apply(get_color)

if fatal_col:
    acled = acled.rename(columns={fatal_col: "fatalities"})
    acled["fatalities"] = pd.to_numeric(acled["fatalities"], errors="coerce").fillna(0)
    acled["marker_size"] = 10 + (acled["fatalities"] / acled["fatalities"].max() * 60)
else:
    acled["marker_size"] = 18

# ---------- 6. PLOT ----------
print("Plotting maps...")
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.patch.set_facecolor("#0f1923")

bounds = [95, 141, -11, 6]

for ax in axes:
    idn.plot(ax=ax, color="#2a3a4a", edgecolor="#5a7a9a", linewidth=0.4)
    ax.set_xlim(bounds[0], bounds[1])
    ax.set_ylim(bounds[2], bounds[3])
    ax.set_facecolor("#0f1923")
    ax.tick_params(colors="white", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#5a7a9a")

# --- Map 1: Fire Heatmap ---
ax1 = axes[0]
fire_cmap = mcolors.LinearSegmentedColormap.from_list(
    "fire", ["#0f1923", "#2d1b00", "#7a2800", "#d94f00", "#ff8c00", "#ffdd00", "#ffffff"])
im = ax1.pcolormesh(XX, YY, ZZ, cmap=fire_cmap, alpha=0.92, shading='auto',
                    norm=mcolors.PowerNorm(gamma=0.4, vmin=ZZ.min(), vmax=ZZ.max()))
cbar1 = fig.colorbar(im, ax=ax1, fraction=0.025, pad=0.02)
cbar1.set_label("Fire Detection Density", color="white", fontsize=10)
cbar1.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar1.ax.yaxis.get_ticklabels(), color="white")
ax1.set_title("Wildfire Detections in Indonesia\n(VIIRS MODIS Satellite Data)", 
              color="white", fontsize=15, fontweight='bold', pad=12)
ax1.set_xlabel("Longitude", color="white", fontsize=10)
ax1.set_ylabel("Latitude", color="white", fontsize=10)

# Annotate key fire hotspot islands
for label, x, y in [("Sumatra", 102.5, 0.5), ("Kalimantan", 114, 1), ("Java", 111, -7.5), ("Sulawesi", 121, -2)]:
    ax1.text(x, y, label, color="white", fontsize=9, alpha=0.7, ha='center',
             fontweight='bold', style='italic')

# --- Map 2: Conflict Events ---
ax2 = axes[1]
ax2.scatter(acled["lon"], acled["lat"],
            c=acled["color"], s=acled["marker_size"],
            alpha=0.6, linewidths=0.2, edgecolors='white')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=9, label=e, linewidth=0)
    for e, c in event_colors.items()
]
if fatal_col:
    legend_elements += [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
               markersize=s, label=f"{'Low' if s==6 else 'High'} fatalities", linewidth=0)
        for s in [6, 12]
    ]
ax2.legend(handles=legend_elements, loc="lower left", fontsize=8,
           facecolor="#1a2a3a", edgecolor="#5a7a9a", labelcolor="white",
           title="Event Type", title_fontsize=9)
ax2.set_title("Conflict Events in Indonesia\n(ACLED Data)", 
              color="white", fontsize=15, fontweight='bold', pad=12)
ax2.set_xlabel("Longitude", color="white", fontsize=10)
ax2.set_ylabel("Latitude", color="white", fontsize=10)

for label, x, y in [("Sumatra", 102.5, 0.5), ("Kalimantan", 114, 1), ("Java", 111, -7.5), ("Sulawesi", 121, -2)]:
    ax2.text(x, y, label, color="white", fontsize=9, alpha=0.7, ha='center',
             fontweight='bold', style='italic')

# ---------- 7. Final layout ----------
fig.suptitle("The Heat Is On: Wildfires and Conflict in Indonesia",
             color="white", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout(pad=2.0)
plt.savefig("output/indonesia_maps.png", dpi=300, bbox_inches="tight",
            facecolor=fig.get_facecolor())
print("Saved: output/indonesia_maps.png")
plt.close()